# Lab 53 — QUBO y Annealing Cuántico

**QUBO** (*Quadratic Unconstrained Binary Optimization*) es el lenguaje nativo de los *annealers* cuánticos como D-Wave. Un problema QUBO busca minimizar:

$$E(\mathbf{x}) = \mathbf{x}^T Q \mathbf{x} = \sum_{i \leq j} Q_{ij} x_i x_j, \quad x_i \in \{0,1\}$$

En este laboratorio:
1. **MAX-CUT** → QUBO: maximizar el corte de un grafo
2. **Portfolio** → QUBO: optimización de Markowitz con restricciones
3. **TSP** → QUBO: problema del viajante con variables binarias
4. **Landscape** energético: visualización de la matriz Q
5. **Comparativa**: solución exacta vs annealing simulado

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import product as iproduct
np.random.seed(42)

def qubo_brute_force(Q, n_vars):
    """Resuelve QUBO por fuerza bruta (exacto para n_vars ≤ 20)."""
    best_x, best_E = None, np.inf
    for bits in iproduct([0,1], repeat=n_vars):
        x = np.array(bits, dtype=float)
        E = x @ Q @ x
        if E < best_E:
            best_E, best_x = E, x.copy()
    return best_x, best_E

def simulated_annealing(Q, n_steps=5000, T_start=2.0, T_end=0.01):
    """Annealing simulado clásico para QUBO."""
    n = Q.shape[0]
    x = np.random.randint(0, 2, n).astype(float)
    E = x @ Q @ x
    best_x, best_E = x.copy(), E
    T = T_start
    energy_history = [E]
    for step in range(n_steps):
        T = T_start * (T_end/T_start) ** (step/n_steps)
        i = np.random.randint(n)
        x_new = x.copy(); x_new[i] = 1 - x_new[i]
        E_new = x_new @ Q @ x_new
        dE = E_new - E
        if dE < 0 or np.random.random() < np.exp(-dE / (T + 1e-10)):
            x, E = x_new, E_new
            if E < best_E:
                best_x, best_E = x.copy(), E
        energy_history.append(E)
    return best_x, best_E, energy_history

print('Funciones QUBO cargadas.')

## 1. MAX-CUT → QUBO

Para un grafo $G=(V,E)$ con pesos $w_{ij}$, MAX-CUT maximiza $\sum_{(i,j)\in E} w_{ij}(x_i+x_j-2x_ix_j)$.
Equivalente QUBO con $Q_{ij} = w_{ij}$, $Q_{ii} = -\sum_j w_{ij}$ (minimizar $-$MAX-CUT).

In [ ]:
# Grafo de 6 nodos
G_mc = nx.cycle_graph(6)
# Añadir aristas adicionales
G_mc.add_edges_from([(0,3),(1,4),(2,5)])
n_mc = G_mc.number_of_nodes()

# Construir matriz QUBO
Q_mc = np.zeros((n_mc, n_mc))
for i, j in G_mc.edges():
    w = 1.0  # peso uniforme
    Q_mc[i,j] += -w
    Q_mc[j,i] += -w
    Q_mc[i,i] += w
    Q_mc[j,j] += w
Q_mc_neg = -Q_mc  # minimizar negativo = maximizar corte

x_bf, E_bf = qubo_brute_force(Q_mc_neg, n_mc)
cut_bf = int(-E_bf)
x_sa, E_sa, hist_sa = simulated_annealing(Q_mc_neg, n_steps=3000)
cut_sa = int(-E_sa)

print(f"MAX-CUT (fuerza bruta):   corte = {cut_bf}, partición = {x_bf.astype(int)}")
print(f"MAX-CUT (annealing sim.): corte = {cut_sa}, partición = {x_sa.astype(int)}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pos = nx.circular_layout(G_mc)
colors_bf = ['steelblue' if x_bf[i] == 0 else 'darkorange' for i in range(n_mc)]
cut_edges = [(i,j) for i,j in G_mc.edges() if x_bf[i] != x_bf[j]]
non_cut   = [(i,j) for i,j in G_mc.edges() if x_bf[i] == x_bf[j]]
nx.draw_networkx(G_mc, pos, ax=axes[0], node_color=colors_bf,
                 edgelist=cut_edges, edge_color='crimson', width=2.5,
                 with_labels=True, node_size=400)
nx.draw_networkx_edges(G_mc, pos, ax=axes[0], edgelist=non_cut, edge_color='gray', width=1)
axes[0].set_title(f'MAX-CUT = {cut_bf} (aristas rojas)'); axes[0].axis('off')

axes[1].imshow(Q_mc_neg, cmap='RdBu', aspect='auto')
plt.colorbar(axes[1].images[0], ax=axes[1])
axes[1].set_title('Matriz QUBO (MAX-CUT negativo)')
axes[1].set_xlabel('Variable j'); axes[1].set_ylabel('Variable i')
plt.tight_layout(); plt.show()

## 2. Portfolio de Markowitz → QUBO

Queremos seleccionar $k$ activos de $n$ disponibles minimizando riesgo (varianza) y maximizando retorno:
$$\min_x \; \lambda \mathbf{x}^T \Sigma \mathbf{x} - (1-\lambda) \boldsymbol{\mu}^T \mathbf{x} + P\left(\mathbf{1}^T\mathbf{x} - k\right)^2$$

In [ ]:
n_assets = 5
k_select = 2  # seleccionar 2 activos
mu = np.array([0.10, 0.15, 0.08, 0.20, 0.12])  # retornos esperados
Sigma_diag = np.array([0.05, 0.12, 0.03, 0.18, 0.07])  # varianzas
# Covarianzas (correlación moderada entre activos)
Sigma = np.diag(Sigma_diag)
Sigma[0,1] = Sigma[1,0] = 0.02
Sigma[2,4] = Sigma[4,2] = 0.01

lam = 0.5   # aversión al riesgo
P   = 1.0   # penalización por violar restricción de cardinalidad

# Construir QUBO
Q_port = lam * Sigma - (1-lam) * np.diag(mu)
# Penalización (∑xi - k)² = ∑xi² + 2∑_{i<j} xixj - 2k∑xi + k²
# En QUBO: agregar P*(1 - 2k) en diagonal y 2P fuera de diagonal
for i in range(n_assets):
    Q_port[i,i] += P * (1 - 2*k_select)
for i in range(n_assets):
    for j in range(i+1, n_assets):
        Q_port[i,j] += 2*P
        Q_port[j,i] += 2*P

x_port, E_port = qubo_brute_force(Q_port, n_assets)
print(f"Portfolio óptimo: activos seleccionados = {np.where(x_port==1)[0].tolist()}")
print(f"  Retorno: {mu @ x_port:.3f}")
print(f"  Riesgo:  {x_port @ Sigma @ x_port:.4f}")
print(f"  k seleccionados: {int(x_port.sum())} (target={k_select})")

# Landscape energético: todas las combinaciones
energies_all = []
labels_all = []
for bits in iproduct([0,1], repeat=n_assets):
    x = np.array(bits, dtype=float)
    E = x @ Q_port @ x
    energies_all.append(E)
    labels_all.append(''.join(map(str,bits)))

fig_port, ax_port = plt.subplots(figsize=(12, 4))
colors_p = ['crimson' if int(lb.count('1'))==k_select else 'steelblue' for lb in labels_all]
ax_port.bar(range(len(energies_all)), energies_all, color=colors_p, width=0.8)
ax_port.set_xlabel('Configuración (binario)'); ax_port.set_ylabel('Energía QUBO')
ax_port.set_title(f'Landscape QUBO Portfolio (rojo = {k_select} activos = factible)')
ax_port.set_xticks(range(0, len(labels_all), 4))
ax_port.set_xticklabels(labels_all[::4], rotation=45, ha='right', fontsize=7)
plt.tight_layout(); plt.show()

## 3. TSP (Problema del Viajante) → QUBO

Para $n$ ciudades, variable $x_{it}=1$ si la ciudad $i$ se visita en el paso $t$. QUBO incluye:
- Restricción de permutación: cada ciudad exactamente una vez, cada paso exactamente una ciudad
- Minimizar distancia total

In [ ]:
n_cities = 4
# Distancias entre ciudades (simétricas)
D = np.array([
    [0, 2, 9, 10],
    [2, 0, 6,  4],
    [9, 6, 0,  3],
    [10,4, 3,  0]
], dtype=float)

A = 10.0  # penalización de restricciones
n_tsp = n_cities * n_cities  # variables: x_{i,t}

def idx(city, step, n):
    return city * n + step

Q_tsp = np.zeros((n_tsp, n_tsp))

# Restricción 1: cada ciudad visitada exactamente una vez → (∑_t x_{i,t} - 1)²
for i in range(n_cities):
    for t in range(n_cities):
        Q_tsp[idx(i,t,n_cities), idx(i,t,n_cities)] -= A
        for t2 in range(t+1, n_cities):
            Q_tsp[idx(i,t,n_cities), idx(i,t2,n_cities)] += 2*A

# Restricción 2: cada paso exactamente una ciudad → (∑_i x_{i,t} - 1)²
for t in range(n_cities):
    for i in range(n_cities):
        Q_tsp[idx(i,t,n_cities), idx(i,t,n_cities)] -= A
        for i2 in range(i+1, n_cities):
            Q_tsp[idx(i,t,n_cities), idx(i2,t,n_cities)] += 2*A

# Objetivo: minimizar distancia total
for i in range(n_cities):
    for j in range(n_cities):
        if i != j:
            for t in range(n_cities):
                t_next = (t+1) % n_cities
                Q_tsp[idx(i,t,n_cities), idx(j,t_next,n_cities)] += D[i,j]

x_tsp, E_tsp = qubo_brute_force(Q_tsp, n_tsp)
route = [int(np.argmax(x_tsp[i*n_cities:(i+1)*n_cities])) for i in range(n_cities)]
total_dist = sum(D[route[t], route[(t+1)%n_cities]] for t in range(n_cities))

print(f"TSP {n_cities} ciudades:")
print(f"  Ruta óptima: {' → '.join(map(str,route))} → {route[0]}")
print(f"  Distancia total: {total_dist:.1f}")

# Benchmark: annealing simulado vs exacto
x_sa_tsp, E_sa_tsp, hist_tsp = simulated_annealing(Q_tsp, n_steps=10000, T_start=10.0)
route_sa = [int(np.argmax(x_sa_tsp[i*n_cities:(i+1)*n_cities])) for i in range(n_cities)]
dist_sa = sum(D[route_sa[t], route_sa[(t+1)%n_cities]] for t in range(n_cities))

fig3, axes3 = plt.subplots(1, 2, figsize=(11, 4))
axes3[0].plot(hist_tsp[::10], color='steelblue', lw=1.5, alpha=0.8)
axes3[0].axhline(E_tsp, color='red', ls='--', lw=2, label=f'Óptimo E={E_tsp:.1f}')
axes3[0].set_xlabel('Iteración (×10)'); axes3[0].set_ylabel('Energía QUBO')
axes3[0].set_title('Annealing simulado — TSP'); axes3[0].legend()

cities_xy = np.array([[0,0],[2,0],[2,3],[0,2]])
for method, route_plot, color in [(f'Exacto ({total_dist:.0f})', route, 'steelblue'),
                                    (f'Annealing ({dist_sa:.0f})', route_sa, 'darkorange')]:
    xs = [cities_xy[c,0] for c in route_plot + [route_plot[0]]]
    ys = [cities_xy[c,1] for c in route_plot + [route_plot[0]]]
    axes3[1].plot(xs, ys, 'o-', label=method, lw=2)
for i, (x,y) in enumerate(cities_xy):
    axes3[1].text(x+0.05, y+0.05, f'C{i}', fontsize=11, fontweight='bold')
axes3[1].set_title('Rutas TSP'); axes3[1].legend(); axes3[1].set_aspect('equal')
plt.tight_layout(); plt.show()

## 4. Comparativa de Solvers QUBO

| Solver | Tipo | Complejidad | Hardware | Notas |
|--------|------|-------------|----------|---------|
| Fuerza bruta | Exacto | O(2ⁿ) | CPU | Solo factible n ≤ 25 |
| Annealing simulado | Heurístico | O(n·steps) | CPU | Solución aproximada |
| D-Wave Advantage | Quantum | Hardware | QPU | 5000+ qubits, Pegasus |
| QAOA p=1 | Quantum | O(n·p) | Circuito | Aproximación |
| Hybrid CQM | Hybrid | variable | Cloud | Mejor calidad, sin límite de tamaño |

Para ejecutar en D-Wave real (requiere token de Leap):
```python
# from dwave.system import DWaveSampler, EmbeddingComposite
# sampler = EmbeddingComposite(DWaveSampler())
# response = sampler.sample_qubo(Q_dict, num_reads=1000)
```

In [ ]:
# Resumen de resultados
print("=" * 50)
print("RESUMEN DE RESULTADOS QUBO")
print("=" * 50)
print(f"MAX-CUT (6 nodos): corte óptimo = {cut_bf} aristas de {G_mc.number_of_edges()}")
print(f"Portfolio (5 activos, k=2): activos {np.where(x_port==1)[0].tolist()}")
print(f"TSP (4 ciudades): ruta {' → '.join(map(str,route))} → {route[0]}, distancia {total_dist:.0f}")
print(f"\nAnnealing simulado encontró el óptimo en TSP: {'Sí' if dist_sa == total_dist else 'No (heurístico)'}")